# Mapas de prospectividad mineral con datos geoespaciales abiertos y machine learning

**Contexto del caso:**
- Punto de partida: un paper de referencia (GIS-based mineral prospectivity mapping, distrito de Tongling, China) que usa 12 mapas predictivos y SVM/ANN/RF sobre depósitos de cobre ya conocidos.
- Reto real: no se contaba con el mismo volumen de depósitos etiquetados que el paper. Había que decidir qué hacer con datos mínimos y abiertos, no con el dataset ideal.
- Se trabajó sobre litología, campo magnético y geoquímica de una zona de estudio, generando una rejilla (grid) uniforme sobre el área de trabajo.
- Artículo completo, con el diagrama del pipeline: **https://fuzzyfrog.ai/es/ai-lab/proyectos/mineria/mapas-prospectividad-mineral-machine-learning/**

**Nota sobre los datos:** los shapefiles de litología, campo magnético y geoquímica de este notebook son de ejemplo. Puedes sustituirlos por tus propios datos abiertos (INEGI, servicios geológicos estatales, USGS) manteniendo el mismo esquema de columnas.


## Diagrama del pipeline

```
Litología (shp) ─┐
Campo magnético ─┼─► Grid uniforme sobre el área de trabajo
Geoquímica ──────┘         │
                  Interpolación por capa
                  (IDW para campo magnético, RBF para geoquímica)
                            │
                  Tabla de condiciones únicas (una fila por celda)
                            │
              ┌─────────────┼─────────────┐
              │             │             │
          KMeans        DBSCAN       Regresión
        (perfiles de   (anomalías   (predicción de
         zona, k=9)    sin asumir   valor geoquímico
                        forma fija) por celda)
              │             │             │
              └─────────────┼─────────────┘
                            │
                Mapa de prospectividad
                (zonas candidatas a explorar)
```

El diagrama interactivo con tooltip por bloque está embebido en el artículo de la plataforma.


## 1. Carga de datos

- Tres capas de entrada: litología (polígonos), campo magnético (puntos con lectura del campo) y geoquímica (puntos con concentración de elementos, ej. Fe, Au).
- Todo se reproyecta a un mismo sistema de coordenadas antes de cualquier operación espacial.
- Reemplaza las rutas por tus propios shapefiles; el esquema de columnas esperado se indica en cada celda.


In [ ]:
import geopandas as gpd

# Reemplaza estas rutas por tus propios shapefiles de litología, campo magnético y geoquímica
litologia_path = "data/litologia.shp"
campo_magnetico_path = "data/campo_magnetico.shp"
geoquimica_path = "data/geoquimica.shp"

gdf_litologia = gpd.read_file(litologia_path).to_crs(epsg=4326)
gdf_geofisica = gpd.read_file(campo_magnetico_path).to_crs(epsg=4326)
gdf_geoquimica = gpd.read_file(geoquimica_path).to_crs(epsg=4326)

print(f"Litología: {len(gdf_litologia)} polígonos")
print(f"Campo magnético: {len(gdf_geofisica)} puntos")
print(f"Geoquímica: {len(gdf_geoquimica)} puntos")


## 2. Explicación de los datos

- La litología define regiones geológicas discretas; se necesita saber en qué unidad litológica cae cada celda de la rejilla.
- El campo magnético y la geoquímica son variables continuas medidas en puntos dispersos: hay que interpolarlas sobre una rejilla uniforme para poder combinarlas con la litología.
- El tamaño de celda es una decisión de ingeniería, no un default: celdas muy grandes diluyen anomalías puntuales, celdas muy pequeñas exceden la resolución real de los datos de entrada.


In [ ]:
from shapely.geometry import box
import numpy as np

# Definir la rejilla uniforme sobre el área combinada de las tres capas
combined_geometry = gdf_litologia.geometry.union_all()
cell_size = 0.02  # grados; ajustar según la resolución real de tus datos de entrada

minx, miny, maxx, maxy = combined_geometry.bounds
rows = int((maxy - miny) / cell_size) + 1
cols = int((maxx - minx) / cell_size) + 1

grid_cells = []
for i in range(rows):
    for j in range(cols):
        x0 = minx + j * cell_size
        y0 = miny + i * cell_size
        cell = box(x0, y0, x0 + cell_size, y0 + cell_size)
        if combined_geometry.intersects(cell):
            grid_cells.append(cell)

grid_gdf = gpd.GeoDataFrame(geometry=grid_cells, crs=gdf_litologia.crs)
print(f"Rejilla generada: {len(grid_gdf)} celdas")


## 3. Ingeniería de variables: unir litología a la rejilla

Condición del proyecto: organizar los datos con la información a priori disponible (la litología conocida) para maximizar la probabilidad de que el modelo encuentre un patrón real, en vez de ruido espacial.


In [ ]:
grid_gdf["centroid"] = grid_gdf.geometry.centroid
litologia_por_celda = gpd.sjoin(
    grid_gdf.set_geometry("centroid"),
    gdf_litologia[["geometry"] + [c for c in gdf_litologia.columns if c not in ("geometry",)][:3]],
    how="left", predicate="within",
)
grid_gdf = grid_gdf.drop(columns=["centroid"])
grid_gdf.head(2)


## 4. Interpolación: dos métodos distintos para dos tipos de señal

**Opción descartada:** usar el mismo método de interpolación (por ejemplo solo IDW) para todas las variables continuas, por simplicidad de código.

**Opción elegida:** IDW (inverse distance weighting) para el campo magnético, porque su variación es relativamente suave y local; RBF (radial basis function, base gaussiana) para la geoquímica, porque su comportamiento espacial es menos regular y se beneficia de una función de base más flexible. Comparar métodos de interpolación es, en la práctica, una forma de divide y vencerás: separar el problema de "qué tan bien capturamos cada señal" del problema de "qué tan bien clasificamos zonas".


In [ ]:
from scipy.spatial import cKDTree

def idw_interpolation(x, y, z, xi, yi, power=2, k=5):
    tree = cKDTree(np.c_[x, y])
    distances, indices = tree.query(np.c_[xi, yi], k=k)
    distances = np.maximum(distances, 1e-10)
    weights = 1 / (distances ** power)
    weights /= weights.sum(axis=1)[:, None]
    return np.sum(weights * z[indices], axis=1)

gdf_geofisica["x"] = gdf_geofisica.geometry.x
gdf_geofisica["y"] = gdf_geofisica.geometry.y
grid_gdf["x"] = grid_gdf.geometry.centroid.x
grid_gdf["y"] = grid_gdf.geometry.centroid.y

# Ajusta "CAMPO_MAG" al nombre real de la columna de lectura magnética en tus datos
grid_gdf["campo_magnetico_idw"] = idw_interpolation(
    gdf_geofisica["x"].values, gdf_geofisica["y"].values,
    gdf_geofisica["CAMPO_MAG"].values, grid_gdf["x"].values, grid_gdf["y"].values,
)


In [ ]:
from scipy.interpolate import Rbf

def rbf_interpolation(gdf_points, columna, grid_gdf, epsg_proyectado=32612):
    puntos = gdf_points.to_crs(epsg=epsg_proyectado)
    rejilla = grid_gdf.to_crs(epsg=epsg_proyectado)
    rbf = Rbf(puntos.geometry.x, puntos.geometry.y, puntos[columna].values, function="gaussian")
    return rbf(rejilla.geometry.centroid.x, rejilla.geometry.centroid.y)

# Ajusta "FE" o "AU" al elemento geoquímico de interés en tus datos
grid_gdf["geoquimica_rbf"] = rbf_interpolation(gdf_geoquimica, "FE", grid_gdf)
grid_gdf[["campo_magnetico_idw", "geoquimica_rbf"]].describe()


## 5. Modelado

Cada celda de la rejilla ahora tiene: litología, campo magnético interpolado y geoquímica interpolada. Con esta tabla de condiciones únicas se entrenan tres enfoques distintos, cada uno respondiendo una pregunta diferente sobre la misma rejilla.


### 5.1 KMeans — perfiles espaciales de la zona

Se usa el índice de silueta para elegir el número de clústeres en vez de fijar un valor arbitrario.


In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

data = grid_gdf.drop(columns=["geometry", "x", "y"]).copy()
data_encoded = pd.get_dummies(data, drop_first=True).fillna(0)

scaler = StandardScaler()
data_scaled = scaler.fit_transform(data_encoded)

silhouette_scores = []
rango_k = range(2, 11)
for k in rango_k:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(data_scaled)
    silhouette_scores.append(silhouette_score(data_scaled, labels))

mejor_k = rango_k[int(np.argmax(silhouette_scores))]
print(f"Mejor número de clústeres según silueta: {mejor_k}")

kmeans = KMeans(n_clusters=mejor_k, random_state=42, n_init=10)
grid_gdf["cluster_kmeans"] = kmeans.fit_predict(data_scaled)


### 5.2 DBSCAN — anomalías sin forma predefinida

KMeans asume clústeres compactos y de forma similar; una anomalía geoquímica real casi nunca tiene esa forma. DBSCAN se usa como contraste, precisamente porque no asume una forma fija.


In [ ]:
from sklearn.cluster import DBSCAN

dbscan = DBSCAN(eps=0.5, min_samples=5)
grid_gdf["cluster_dbscan"] = dbscan.fit_predict(data_scaled)

n_anomalias = (grid_gdf["cluster_dbscan"] == -1).sum()
print(f"Celdas marcadas como anomalía (ruido/outlier) por DBSCAN: {n_anomalias}")


### 5.3 Regresión — predicción del valor geoquímico por celda

Se comparan varios regresores en vez de fijar uno solo de entrada, porque no hay garantía previa de que la relación entre litología, campo magnético y geoquímica sea lineal.


In [ ]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

X = data_encoded.drop(columns=["geoquimica_rbf"], errors="ignore")
y = grid_gdf["geoquimica_rbf"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

modelos = {
    "Regresión lineal": LinearRegression(),
    "Ridge": Ridge(),
    "Lasso": Lasso(),
    "Árbol de decisión": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
    "SVR": SVR(),
}

resultados = []
for nombre, modelo in modelos.items():
    modelo.fit(X_train, y_train)
    pred = modelo.predict(X_test)
    resultados.append({
        "modelo": nombre,
        "rmse": mean_squared_error(y_test, pred, squared=False),
        "r2": r2_score(y_test, pred),
    })

pd.DataFrame(resultados).sort_values("r2", ascending=False)


## 6. Evaluación: mapa de prospectividad combinado

El mapa final no depende de un único modelo. Se combinan las tres señales, perfil de KMeans, anomalía de DBSCAN y valor predicho por el mejor regresor, en un puntaje simple por celda. Esto es intencional: cada modelo aporta una pregunta distinta y el mapa de prospectividad es más honesto si conserva esa separación en vez de esconderla detrás de un solo número.


In [ ]:
grid_gdf["es_anomalia_dbscan"] = (grid_gdf["cluster_dbscan"] == -1).astype(int)

mejor_modelo_nombre = pd.DataFrame(resultados).sort_values("r2", ascending=False).iloc[0]["modelo"]
mejor_modelo = modelos[mejor_modelo_nombre]
grid_gdf["geoquimica_predicha"] = mejor_modelo.predict(X)

# Puntaje simple de prospectividad: geoquímica predicha normalizada + bono por anomalía DBSCAN
geoq_norm = (grid_gdf["geoquimica_predicha"] - grid_gdf["geoquimica_predicha"].min()) / (
    grid_gdf["geoquimica_predicha"].max() - grid_gdf["geoquimica_predicha"].min()
)
grid_gdf["puntaje_prospectividad"] = geoq_norm + 0.3 * grid_gdf["es_anomalia_dbscan"]

grid_gdf[["cluster_kmeans", "es_anomalia_dbscan", "geoquimica_predicha", "puntaje_prospectividad"]].describe()


## 7. Hallazgos principales

- Con datos mínimos y abiertos, en vez del set de depósitos etiquetados del paper de referencia, un enfoque no supervisado (KMeans y DBSCAN) más regresión resultó más viable que forzar una clasificación supervisada tipo SVM/ANN/RF sin suficientes ejemplos positivos.
- Usar dos métodos de interpolación distintos, IDW para campo magnético y RBF para geoquímica, capturó mejor el comportamiento real de cada señal que forzar un único método para todo.
- DBSCAN aportó algo que KMeans no puede dar por diseño: anomalías sin forma predefinida, que en prospección mineral suelen ser justo las zonas más interesantes.
- El puntaje de prospectividad combinado es deliberadamente simple y transparente, para que cualquier geólogo pueda auditar por qué una celda quedó marcada como candidata, en vez de confiar en una caja negra.
